In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from scipy.sparse import hstack

# Load datasets
train_df = pd.read_csv("/kaggle/input/data-quest-2025-sboui-special-challenge/train_Set.csv")
test_df = pd.read_csv("/kaggle/input/data-quest-2025-sboui-special-challenge/Test_Set.csv")

# Handle missing values
train_df["Comments"] = train_df["Comments"].fillna("")
test_df["Comments"] = test_df["Comments"].fillna("")
train_df.fillna(0, inplace=True)
test_df.fillna(0, inplace=True)

# Define Tunisian stop words
tunisian_stopwords = set([
    "w", "7ata", "we", "wa", "3la", "ou", "el", "a", "ya", "y", "ti", "mil", 
    "e", "3ala", "mn", "men", "la", "le", "bara", "ki", "k", "o", "kol", "li", "l", "bi", "b"
])

# Remove punctuation
PUNCT_TO_REMOVE = string.punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', PUNCT_TO_REMOVE))

# Remove Tunisian stopwords
def remove_stopwords(text):
    return " ".join([word for word in text.split() if word.lower() not in tunisian_stopwords])

# Basic stemming for Tunisian dialect
def stem_tunisian(text):
    replacements = {
        "y": "", "ya": "",  # Remove common suffixes
        "el": "", "ti": "", "7ata": "", "we": ""
    }
    words = text.split()
    return " ".join([replacements.get(word, word) for word in words])

# Apply text preprocessing
def preprocess_tunisian_text(text):
    text = remove_punctuation(text)
    text = remove_stopwords(text)
    text = stem_tunisian(text)
    return text

# Apply to dataset
train_df["Cleaned_Comments"] = train_df["Comments"].apply(preprocess_tunisian_text)
test_df["Cleaned_Comments"] = test_df["Comments"].apply(preprocess_tunisian_text)

# Feature Engineering
train_df["word_count"] = train_df["Comments"].apply(lambda x: len(x.split()))
train_df["char_count"] = train_df["Comments"].apply(lambda x: len(x))
train_df["avg_word_length"] = train_df["char_count"] / train_df["word_count"]

def sentiment_features(text):
    positive_words = {"bravo", "mrigel", "chapeau"}  # Positive words in Tunisian dialect
    religious_words = {"rabi", "allah", "inchallah", "hamdoulah"}
    name_words = {"sbou3i", "sab3oun"}
    conversational_words = {"fi", "ya", "el", "si", "tawa", "ahla"}
    
    words = text.lower().split()
    return (
        sum(1 for word in words if word in positive_words),
        sum(1 for word in words if word in religious_words),
        sum(1 for word in words if word in name_words),
        sum(1 for word in words if word in conversational_words),
    )

train_df["pos_count"], train_df["religious_count"], train_df["name_count"], train_df["conv_count"] = zip(*train_df["Comments"].apply(sentiment_features))

# TF-IDF Vectorization
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=500, stop_words=list(tunisian_stopwords))
X_tfidf = tfidf.fit_transform(train_df["Cleaned_Comments"])

# Combine all features
X_combined = hstack([
    X_tfidf,
    train_df[["word_count", "char_count", "avg_word_length", "pos_count", "religious_count", "name_count", "conv_count"]].values
])

y = train_df["Revenue"]

# Train-Test Split
X_train, X_val, y_train, y_val = train_test_split(X_combined, y, test_size=0.2, random_state=42)

# Train Models
xgb_model = XGBRegressor(n_estimators=150, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_val)
r2_xgb = r2_score(y_val, y_pred_xgb)
print(f"XGBoost R² Score: {r2_xgb}")

# rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
# rf_model.fit(X_train, y_train)
# y_pred_rf = rf_model.predict(X_val)
# r2_rf = r2_score(y_val, y_pred_rf)
# print(f"RandomForest R² Score: {r2_rf}")

# ridge_model = Ridge(alpha=1.0)
# ridge_model.fit(X_train, y_train)
# y_pred_ridge = ridge_model.predict(X_val)
# r2_ridge = r2_score(y_val, y_pred_ridge)
# print(f"Ridge Regression R² Score: {r2_ridge}")

# Baseline Mean Model
mean_revenue = train_df["Revenue"].mean()
y_pred_mean = np.full_like(y_val, mean_revenue)
r2_mean = r2_score(y_val, y_pred_mean)
print(f"Baseline Mean Prediction R² Score: {r2_mean}")

# Prepare Test Set Features
test_df["word_count"] = test_df["Comments"].apply(lambda x: len(x.split()))
test_df["char_count"] = test_df["Comments"].apply(lambda x: len(x))
test_df["avg_word_length"] = test_df["char_count"] / test_df["word_count"]
test_df["pos_count"], test_df["religious_count"], test_df["name_count"], test_df["conv_count"] = zip(*test_df["Comments"].apply(sentiment_features))

X_test_tfidf = tfidf.transform(test_df["Cleaned_Comments"])
X_test_combined = hstack([
    X_test_tfidf,
    test_df[["word_count", "char_count", "avg_word_length", "pos_count", "religious_count", "name_count", "conv_count"]].values
])

# # Generate Predictions
# test_df["Revenue_XGBoost"] = xgb_model.predict(X_test_combined)
# test_df["Revenue_RandomForest"] = rf_model.predict(X_test_combined)
# test_df["Revenue_Ridge"] = ridge_model.predict(X_test_combined)

# # Ensemble: Weighted Average of Models
# test_df["Revenue"] = (0.5 * test_df["Revenue_XGBoost"] + 
#                       0.3 * test_df["Revenue_RandomForest"] + 
#                       0.2 * test_df["Revenue_Ridge"])

# # Save Submission
# test_df[["ID", "Revenue"]].to_csv("final_submission.csv", index=False)


XGBoost R² Score: -0.002059526845812787
Baseline Mean Prediction R² Score: -0.00019045806085804529


In [2]:
mean_revenue = train_df["Revenue"].mean()

# Assign Mean Revenue as the Prediction for Test Set
test_df["Revenue"] = mean_revenue

# Save Submission (ID + Revenue)
test_df[["ID", "Revenue"]].to_csv("final_submission.csv", index=False)

print(f"Submission saved with mean revenue: {mean_revenue}")

Submission saved with mean revenue: 4023.7234062535663
